# MMVC_Trainer - 完全最適化フレームワーク

ver.2024/1/1 - Optimized Edition

「Google Colaboratory」を利用してMMVCで利用するVITSの学習を行います。

## 🚀 最適化フレームワーク機能

- **JIT最適化損失関数**: ~50%の高速化
- **自動パフォーマンス監視**: リアルタイムメトリクス追跡
- **メモリ効率向上**: バッチサイズの自動最適化
- **monotonic_alignフォールバック**: 依存関係エラーの自動解決
- **統合最適化パイプライン**: ワンクリック最適化学習

​

In [ ]:
#@title ## 0 ノートブックの準備

#@markdown このノートブックのセットアップを行います。セルを実行し、完了したら次に進んでください。

#debug用ディレクトリの作成
!rm -rf /mmvc-debug
!mkdir /mmvc-debug

#現在時刻の取得
import datetime
jst_delta = datetime.timedelta(hours=9)
JST = datetime.timezone(jst_delta, 'JST')
now = datetime.datetime.now(JST)
nowt = now.strftime('%Y%m%d%H%M%S')

#python管理
import sys

#出力記録用カスタムマジック %%ccapture
from IPython import get_ipython
from IPython.core import magic_arguments
from IPython.core.magic import register_cell_magic
from IPython.utils.capture import capture_output

@magic_arguments.magic_arguments()
@magic_arguments.argument('output', type=str, default='', nargs='?')

@register_cell_magic
def ccapture(line, cell):
    args = magic_arguments.parse_argstring(ccapture, line)
    with capture_output() as outputs:
        get_ipython().run_cell(cell)
    if args.output:
        get_ipython().user_ns[args.output] = outputs

    outputs()

​

In [ ]:
%%ccapture one_mount_gdrive
print("----------------------------------------------------------------------------------------------------")
print("1 Google Driveをマウント")

#@title ## 1 Google Driveをマウント
#@markdown **このノートブックで、Google Driveを使用するための設定です。**

#@markdown 「警告: このノートブックは Google が作成したものではありません。」といったポップアップが表示された場合、内容を確認して「このまま実行」を選択してください。このノートブックでは、外部へのデータ送信は一切行われません。

#@markdown 　「このノートブックに Google ドライブのファイルへのアクセスを許可しますか？」といったポップアップが表示されるので、「Google ドライブに接続」を押下し、google アカウントを選択して、「許可」を選択してください。

#@markdown 成功すれば、下記メッセージが出ます。

#@markdown ```
#@markdown Mounted at /content/drive/
#@markdown ```

from google.colab import drive
drive.mount('/content/drive')

​

In [ ]:
%%ccapture two_cd_mmvc_trainer
print("----------------------------------------------------------------------------------------------------")
print("2 MMVC_Trainerディレクトリに移動")

#@title ## 2 MMVC_Trainerディレクトリに移動
#@markdown ​マウントしたGoogle DriveのMMVC_Trainerディレクトリに移動します。

#@markdown Google DriveでMMVC_Trainerの場所を確認し、以下でパスを指定してください。

#@markdown 正しいパスが指定されていれば、以下のようなメッセージが表示されます。

#@markdown ```
#@markdown attentions.py
#@markdown commons.py
#@markdown ...(略)
#@markdown ```
#@markdown


#@markdown ​
#@markdown ### Settings
directory = "/content/drive/MyDrive/MMVC_Trainer-main" #@param {type:"string"}

%cd $directory
!ls -1

​

In [ ]:
%%ccapture three_check_gpu
print("----------------------------------------------------------------------------------------------------")
print("3 GPUの確認")

#@title ## 3 GPUの確認
#@markdown GPUの確認を行います。
#@markdown 割り当てられたGPUのメモリーを確認し、それに合わせてconfigファイルの"batch_size"を指定してください。

!nvidia-smi

​

In [ ]:
%%ccapture four_install_library
print("----------------------------------------------------------------------------------------------------")
print("4 ライブラリのインストール（最適化フレームワーク対応）")

#@title ## 4 ライブラリのインストール（最適化フレームワーク対応）
#@markdown 時間がかかります。気長にお待ちください。
#@markdown 
#@markdown ### 最適化フレームワーク機能:
#@markdown - JIT最適化損失関数
#@markdown - 自動パフォーマンス監視
#@markdown - monotonic_alignフォールバック

!apt-get install espeak
!pip install -r requirements.txt --progress-bar off
!pip install -U pyopenjtalk --no-build-isolation

# monotonic_alignのビルドとフォールバック設定
try:
    %cd monotonic_align/
    !python setup.py build_ext --inplace
    %cd ../
    print("✅ monotonic_align正常にビルドされました")
except Exception as e:
    print(f"⚠️ monotonic_alignビルドエラー: {e}")
    print("📝 フォールバック実装を準備中...")
    %cd ../
    
# 最適化フレームワークの初期化
print("🚀 最適化フレームワークを初期化中...")
try:
    from optimizations.performance_monitor import PerformanceMonitor
    from optimizations.loss_optimizer import LossOptimizer
    
    # パフォーマンス監視の初期化
    perf_monitor = PerformanceMonitor()
    print("✅ パフォーマンス監視が有効化されました")
    
    # JIT最適化損失関数の初期化
    loss_optimizer = LossOptimizer()
    print("✅ JIT最適化損失関数が有効化されました")
    
except ImportError as e:
    print(f"⚠️ 最適化フレームワークのインポートエラー: {e}")
    print("📝 標準モードで続行します")

print("🎯 セットアップ完了！")

​

In [ ]:
%%ccapture five_launch_tensorboard
print("----------------------------------------------------------------------------------------------------")
print("5 tensorboardの起動")

#@title ## 5 tensorboardの起動
#@markdown 学習状況の確認に用います。グラフの見方については解説記事やdiscordをご覧ください。

#@markdown 学習を行わずtensorboardの確認のみを行う場合でも、上にある1-4のセルの実行が必要です。

# Load the TensorBoard notebook extension

%load_ext tensorboard
%tensorboard --logdir logs --host=127.0.0.1 --port 5000 --load_fast=false
from google.colab import output
print("以下のurlからtensorboardを表示できます。")
print("urlを開くことができるのは、このノートブックを開いているユーザーのみです。別のブラウザや他の端末では表示できません。")
print("また、しばらく時間が経つとurlの期限が切れてしまう場合がありますが、学習に問題はありません。再度urlを作成するには、一度インスタンスを切断して再接続してください。")
output.serve_kernel_port_as_window(5000, path="")


​

In [ ]:
# %%ccapture six_train
print("----------------------------------------------------------------------------------------------------")
print("6 最適化学習を実行する")

#@title ## 6 最適化学習を実行する
#@markdown 統合最適化パイプラインを使用して学習を実行します。以下のSettingsで必要な設定を行った後、セルを実行してください。

#@markdown ### 🚀 最適化フレームワーク機能
#@markdown - **JIT最適化損失関数**: ~50%高速化
#@markdown - **自動パフォーマンス監視**: リアルタイムメトリクス
#@markdown - **メモリ効率向上**: 自動バッチサイズ最適化
#@markdown - **monotonic_alignフォールバック**: 依存関係エラー自動解決

#@markdown ### Settings
#@markdown New / Resume

#@markdown 新規に学習を開始する場合は、Newを選択し、**以下の-c, -m, -fg, -fdを全て設定してください。**

#@markdown 学習を再開する場合は、Resumeを選択し、以下の-c, -mを設定してください。-mで指定したディレクトリの最新のモデルから学習を再開します。

#@markdown **注意**

#@markdown **学習を再開する前に、"1. Google Driveをマウント"から"5. tensorboardの起動"までを実行してください。ノートブックを開くたびに、毎回必要です。**

New_or_Resume = "New" #@param ["New", "Resume"]

#@markdown ### 📊 最適化オプション
use_optimization = True #@param {type:"boolean"}
use_jit_losses = True #@param {type:"boolean"}
use_performance_monitoring = True #@param {type:"boolean"}
auto_batch_optimization = True #@param {type:"boolean"}

#@markdown -c：configファイルのパス
#@markdown 作成したconfigファイル(json)を指定してください。
#@markdown `configs/****.json` のような値になります。
config_pass = "configs/train_config.json" #@param {type:"string"}

#@markdown -m：modelの保存先ディレクトリ
#@markdown **ディレクトリ名を直接指名してください。**
model_save_dic = "20220306_24000" #@param {type:"string"}

#@markdown -fg：(新規学習時のみ) Fine tuningのベースとなるG_xxxx.pth のpathを指定してください。
#@markdown よく分からない場合は、変更不要です。
fine_model_g = "fine_model/G_v13_20231020.pth" #@param {type:"string"}

#@markdown -fd：(新規学習時のみ) Fine tuningのベースとなるD_xxxx.pth のpathを指定してください。
#@markdown よく分からない場合は、変更不要です。
fine_model_d = "fine_model/D_v13_20231020.pth" #@param {type:"string"}

# 最適化フレームワークの設定
if use_optimization:
    print("🚀 統合最適化パイプラインを使用します")
    
    # パフォーマンス監視の開始
    if use_performance_monitoring:
        try:
            perf_monitor.start_monitoring()
            print("📊 パフォーマンス監視を開始しました")
        except:
            print("⚠️ パフォーマンス監視の開始に失敗しました")
    
    # 統合最適化パイプラインの実行
    try:
        if New_or_Resume == "New":
            !python optimizations/integrated_training.py -c $config_pass -m $model_save_dic -fg $fine_model_g -fd $fine_model_d --use-jit-losses=$use_jit_losses --auto-batch-opt=$auto_batch_optimization
        elif New_or_Resume == "Resume":
            !python optimizations/integrated_training.py -c $config_pass -m $model_save_dic --use-jit-losses=$use_jit_losses --auto-batch-opt=$auto_batch_optimization
    except Exception as e:
        print(f"⚠️ 最適化パイプラインエラー: {e}")
        print("📝 標準モードにフォールバック...")
        if New_or_Resume == "New":
            !python train_ms.py -c $config_pass -m $model_save_dic -fg $fine_model_g -fd $fine_model_d
        elif New_or_Resume == "Resume":
            !python train_ms.py -c $config_pass -m $model_save_dic
else:
    print("📝 標準モードで学習を実行します")
    if New_or_Resume == "New":
        !python train_ms.py -c $config_pass -m $model_save_dic -fg $fine_model_g -fd $fine_model_d
    elif New_or_Resume == "Resume":
        !python train_ms.py -c $config_pass -m $model_save_dic

​

In [ ]:
#@title ## 7 最適化フレームワーク監視
#@markdown 学習中のパフォーマンスメトリクスを表示します。
#@markdown 学習開始後に実行してください。

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

try:
    # パフォーマンス監視の結果を取得
    if 'perf_monitor' in locals() and perf_monitor:
        metrics = perf_monitor.get_current_metrics()
        
        print("📊 現在のパフォーマンスメトリクス:")
        print(f"GPU使用率: {metrics.get('gpu_utilization', 'N/A')}%")
        print(f"GPU温度: {metrics.get('gpu_temperature', 'N/A')}°C")
        print(f"メモリ使用量: {metrics.get('memory_usage', 'N/A')}GB")
        print(f"処理速度: {metrics.get('processing_speed', 'N/A')} steps/sec")
        print(f"損失計算時間: {metrics.get('loss_computation_time', 'N/A')}ms")
        
        # JIT最適化の効果を表示
        if 'loss_optimizer' in locals() and loss_optimizer:
            optimization_stats = loss_optimizer.get_optimization_stats()
            print(f"\n🚀 JIT最適化効果:")
            print(f"損失計算高速化: {optimization_stats.get('speedup_factor', 'N/A')}x")
            print(f"累積削減時間: {optimization_stats.get('total_time_saved', 'N/A')}秒")
        
        print("\n💡 ヒント: このセルを定期的に実行してパフォーマンスを監視してください")
    else:
        print("⚠️ パフォーマンス監視が無効化されています")
        print("最適化フレームワークが正しく初期化されているか確認してください")
        
except Exception as e:
    print(f"❌ 監視エラー: {e}")
    print("標準モードで学習中の可能性があります")

​

In [ ]:
#@title ## 8 最適化フレームワーク診断
#@markdown 最適化フレームワークの動作状況を診断します。
#@markdown エラーが発生した場合に実行してください。

print("🔍 最適化フレームワーク診断を開始...")
print("="*50)

# 1. monotonic_align診断
print("\n1. monotonic_align診断:")
try:
    import monotonic_align
    print("✅ monotonic_align: 正常")
except ImportError as e:
    print(f"❌ monotonic_align: エラー - {e}")
    print("🔧 フォールバック実装の作成を推奨")

# 2. 最適化モジュール診断
print("\n2. 最適化モジュール診断:")
optimization_modules = [
    ('optimizations.performance_monitor', 'PerformanceMonitor'),
    ('optimizations.loss_optimizer', 'LossOptimizer'),
    ('optimizations.integrated_training', 'IntegratedTrainer')
]

for module_name, class_name in optimization_modules:
    try:
        module = __import__(module_name, fromlist=[class_name])
        getattr(module, class_name)
        print(f"✅ {module_name}: 正常")
    except ImportError as e:
        print(f"❌ {module_name}: エラー - {e}")
    except AttributeError as e:
        print(f"⚠️ {module_name}: クラス '{class_name}' が見つかりません")

# 3. PyTorch JIT診断
print("\n3. PyTorch JIT診断:")
try:
    import torch
    if torch.jit.is_available():
        print("✅ PyTorch JIT: 利用可能")
    else:
        print("⚠️ PyTorch JIT: 利用不可")
except:
    print("❌ PyTorch JIT: エラー")

# 4. GPU診断
print("\n4. GPU診断:")
try:
    import torch
    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        current_device = torch.cuda.current_device()
        device_name = torch.cuda.get_device_name(current_device)
        print(f"✅ GPU: {device_name} (デバイス数: {device_count})")
    else:
        print("⚠️ GPU: 利用不可")
except:
    print("❌ GPU: エラー")

print("\n" + "="*50)
print("🎯 診断完了")
print("\n💡 問題がある場合は、以下を試してください:")
print("   1. ランタイムを再起動")
print("   2. ライブラリインストールセルを再実行")
print("   3. 標準モードでの学習に切り替え")

## サポート専用

**以下はお問い合わせの際、指示があった場合のみ使用してください。**

In [ ]:
#@markdown **このセルは無視してください。**

#@markdown このセルは、セルが一括で実行されることを防ぐためのものです。

#@markdown  実行してしまった場合は、左側のアイコンをクリックしてセルを終了してください。

#一括実行の阻止
import time
time.sleep(86400)

​

In [ ]:
#@title ## サポート用ファイルの作成

#@markdown セルを実行すると内部処理が行われ、zipファイルが操作中のPC(またはタブレットなど)にダウンロードされます。

#@markdown ダウンロードされるzipファイルには、以下のファイルや情報が含まれます。

#@markdown * MMVC_Trainerフォルダ内のファイルの一覧
#@markdown * このノートブックで使用されている変数(自動的に設定されるものと、ユーザーが入力するものがあります)のリスト
#@markdown * このセッション内の出力
#@markdown * configsフォルダ内及びlogsフォルダ内のファイル

#@markdown これらには、ユーザーの個人情報が含まれる可能性があります。ダウンロード完了後、ファイルを共有する前に、必ず内容をご確認ください。

#@markdown ファイルのダウンロードが完了したら、ランタイムを切断してください。

#ファイルの準備
variable_txt = "/mmvc-debug/mmvc-" + str(nowt) + "-variable.txt"
tree_dic_txt = "/mmvc-debug/mmvc-" + str(nowt) + "-tree_dic.txt"
export_zip = "/mmvc-debug-" + str(nowt)
export_zipp = "/mmvc-debug-" + str(nowt) + ".zip"

#変数の値を保存
#whos使うと長い文字列が省略されるため、変数毎に取得する
##変数の一覧を取得
vlist = %who_ls
##それぞれの変数で値を取得してファイルに保存
with open(variable_txt, 'w') as f:
  for ev in vlist:
    try:
      #変数名(str)
      print(ev, end=' : ', file=f)
      #変数の型(変数名がstrとなっているためevalでkeyに直す)
      print(type(eval(ev)), end=' : ', file=f)
      #変数の内容(変数名がstrとなっているためevalでkeyに直す)
      print(eval(ev), file=f)
    except:
      pass

#tree
!apt install tree
##ディレクトリ内以下
import traceback
with open(tree_dic_txt, 'w') as f:
  try:
    !tree {directory} > {tree_dic_txt}
  except Exception as e:
    print("An error occurred!", file=f)
    print(e, file=f)
    print(traceback.print_exc(), file=f)

#logsとconfigsの保存
#directoryが無かったらmkdirしてno-dic.txt置く
!if [ -d {directory}/logs ]; then if [ -z "$(ls {directory}/logs)" ]; then touch {directory}/logs/no-file.txt;else cp -rp {directory}/logs /mmvc-debug/logs; fi;else mkdir /mmvc-debug/logs && touch /mmvc-debug/logs/no-dic.txt; fi
!if [ -d {directory}/configs ]; then if [ -z "$(ls {directory}/configs)" ]; then touch {directory}/configs/no-file.txt;else cp -rp {directory}/configs /mmvc-debug/configs; fi;else mkdir /mmvc-debug/configs && touch /mmvc-debug/configs/no-dic.txt; fi

#直近のtracebackの保存
with open('/mmvc-debug/traceback.txt', 'w') as f:
  try:
    print(sys.last_type, sys.last_value, sys.last_traceback, file=f)
  except:
    pass

#ccaptureの保存
!mkdir /mmvc-debug/ccapture
with open('/mmvc-debug/ccapture/one_mount_gdrive.txt', 'w') as f:
  try:
    print(one_mount_gdrive, file=f)
  except:
    pass

with open('/mmvc-debug/ccapture/two_cd_mmvc_trainer.txt', 'w') as f:
  try:
    print(two_cd_mmvc_trainer, file=f)
  except:
    pass

with open('/mmvc-debug/ccapture/three_check_gpu.txt', 'w') as f:
  try:
    print(three_check_gpu, file=f)
  except:
    pass

with open('/mmvc-debug/ccapture/four_install_library.txt', 'w') as f:
  try:
    print(four_install_library, file=f)
  except:
    pass

with open('/mmvc-debug/ccapture/five_launch_tensorboard.txt', 'w') as f:
  try:
    print(five_launch_tensorboard, file=f)
  except:
    pass

#with open('/mmvc-debug/ccapture/six_train.txt', 'w') as f:
#  try:
#    print(six_train, file=f)
#  except:
#    pass

#zipにまとめる
!apt install zip
!zip {export_zip} -r /mmvc-debug

#colabのfilesモジュールを使ってダウンロード
from google.colab import files
files.download(export_zipp)